In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mysql.connector

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

print("All libraries imported successfully!")


All libraries imported successfully!


In [2]:
connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Nishant@123",
    database="car_price_prediction"
)

print("MySQL connection successful!")

MySQL connection successful!


In [3]:
query = """
SELECT *
FROM cars;
"""

df = pd.read_sql(query, connection)

df.head()

C:\Users\HP\AppData\Local\Temp\ipykernel_22968\2741079312.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, connection)


,id,car_name,year,selling_price,km_driven,fuel,seller_type,transmission,owner
0,1,Maruti 800 AC,2007,60000.0,70000,Petrol,Individual,Manual,First Owner
1,2,Maruti Wagon R LXI Minor,2007,135000.0,50000,Petrol,Individual,Manual,First Owner
2,3,Hyundai Verna 1.6 SX,2012,600000.0,100000,Diesel,Individual,Manual,First Owner
3,4,Datsun RediGO T Option,2017,250000.0,46000,Petrol,Individual,Manual,First Owner
4,5,Honda Amaze VX i-DTEC,2014,450000.0,141000,Diesel,Individual,Manual,Second Owner


In [4]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 3577
Columns: 9


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3577 entries, 0 to 3576
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             3577 non-null   int64  
 1   car_name       3577 non-null   str    
 2   year           3577 non-null   int64  
 3   selling_price  3577 non-null   float64
 4   km_driven      3577 non-null   int64  
 5   fuel           3577 non-null   str    
 6   seller_type    3577 non-null   str    
 7   transmission   3577 non-null   str    
 8   owner          3577 non-null   str    
dtypes: float64(1), int64(3), str(5)
memory usage: 452.3 KB


In [6]:
df.isnull().sum()

id               0
car_name         0
year             0
selling_price    0
km_driven        0
fuel             0
seller_type      0
transmission     0
owner            0
dtype: int64

In [7]:
df.describe()

,id,year,selling_price,km_driven
count,3577.000000,3577.000000,3.577000e+03,3577.000000
mean,2087.193458,2012.962538,4.739125e+05,69250.545709
std,1254.720507,4.251759,5.093018e+05,47579.940016
min,1.000000,1992.000000,2.000000e+04,1.000000
25%,1026.000000,2010.000000,2.000000e+05,36000.000000
50%,2012.000000,2013.000000,3.500000e+05,60000.000000
75%,3193.000000,2016.000000,6.000000e+05,90000.000000
max,4340.000000,2020.000000,8.900000e+06,806599.000000


In [8]:
X=df.drop("selling_price",axis=1)
y=df["selling_price"]

In [9]:
print("Input features:")
print(X.columns)

print("\nTarget:")
print(y.name)

Input features:
Index(['id', 'car_name', 'year', 'km_driven', 'fuel', 'seller_type',
       'transmission', 'owner'],
      dtype='str')

Target:
selling_price


In [10]:
X = X.drop("id", axis=1)

In [11]:
numeric_features = [
    "year",
    "car_age",
    "km_driven"
]

categorical_features = [
    "fuel",
    "seller_type",
    "transmission",
    "owner"
]

In [12]:
X = pd.get_dummies(
    X,
    columns=categorical_features,
    drop_first=True
)

In [13]:
X.head()

,car_name,year,km_driven,fuel_Diesel,fuel_Electric,fuel_LPG,fuel_Petrol,seller_type_Individual,seller_type_Trustmark Dealer,transmission_Manual,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Maruti 800 AC,2007,70000,False,False,False,True,True,False,True,False,False,False,False
1,Maruti Wagon R LXI Minor,2007,50000,False,False,False,True,True,False,True,False,False,False,False
2,Hyundai Verna 1.6 SX,2012,100000,True,False,False,False,True,False,True,False,False,False,False
3,Datsun RediGO T Option,2017,46000,False,False,False,True,True,False,True,False,False,False,False
4,Honda Amaze VX i-DTEC,2014,141000,True,False,False,False,True,False,True,False,True,False,False


In [14]:
print(X.shape)

(3577, 14)


In [15]:
X = X.drop("car_name", axis=1)

In [16]:
X = X.astype(float)

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [18]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [19]:
print(tf.__version__)

2.22.0-rc0


In [76]:
model = Sequential([
    Dense(599, activation="relu"),
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1)
])

C:\Users\HP\AppData\Roaming\Python\Python314\site-packages\keras\src\layers\core\dense.py:102: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [77]:
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=['mse']
)

In [78]:
model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_33 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_34 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_35 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_36 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_37 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [79]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

In [80]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=1000,
    batch_size=52,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/1000
44/44 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - loss: 452674715648.0000 - mse: 452674715648.0000 - val_loss: 519668367360.0000 - val_mse: 519668367360.0000
Epoch 2/1000
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 451804430336.0000 - mse: 451804430336.0000 - val_loss: 516356734976.0000 - val_mse: 516356734976.0000
Epoch 3/1000
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 437982560256.0000 - mse: 437982560256.0000 - val_loss: 479011405824.0000 - val_mse: 479011405824.0000
Epoch 4/1000
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 354782117888.0000 - mse: 354782117888.0000 - val_loss: 319672352768.0000 - val_mse: 319672352768.0000
Epoch 5/1000
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 188499836928.0000 - mse: 188499836928.0000 - val_loss: 179648446464.0000 - val_mse: 179648446464.0000
Epoch 6/1000
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 137755410432.0000 - mse: 137755410432.0000 - val_loss: 166240731136.0000 - val_mse: 166240731136.0000
Epoch 7/1000
44/44 ━━━━━━━━

In [81]:
y_pred = model.predict(X_test)

23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step


In [82]:
y_pred = y_pred.flatten()

In [83]:
print(y_pred[:10])

[5.0457312e+05 5.2203403e+05 6.7420500e+05 2.2339835e+06 1.5098125e+05
 2.6150536e+05 1.3959131e+05 7.1168719e+05 9.7887806e+05 3.1948866e+05]


In [28]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_test, y_pred)

print("MAE:", mae)

MAE: 185071.20603286487


In [29]:
from sklearn.metrics import mean_squared_error

mse = mean_squared_error(y_test, y_pred)

print("MSE:", mse)

MSE: 169845556630.98303


In [84]:
from sklearn.metrics import r2_score

r2=r2_score(y_test , y_pred)
r2

0.49259162511741106

In [31]:
rmse = np.sqrt(mse)

print("RMSE:", rmse)

RMSE: 412123.2299094326


In [32]:
print("Model Evaluation")
print("---------------------------")
print("MAE  :", mae)
print("MSE  :", mse)
print("RMSE :", rmse)
print("R²   :", r2)

Model Evaluation
---------------------------
MAE  : 185071.20603286487
MSE  : 169845556630.98303
RMSE : 412123.2299094326
R²   : 0.4727476858286782


In [33]:
results = pd.DataFrame({
    "Actual Price": y_test.values,
    "Predicted Price": y_pred
})

results.head(20)

,Actual Price,Predicted Price
0,270000.0,4.974250e+05
1,525000.0,4.669639e+05
2,990000.0,6.799097e+05
3,3800000.0,1.949911e+06
4,229999.0,1.410690e+05
5,325000.0,2.855185e+05
6,210000.0,1.426002e+05
7,500000.0,7.163642e+05
8,790000.0,9.647753e+05
9,360000.0,2.851278e+05


In [34]:
results["Error"] = (
    results["Actual Price"] - results["Predicted Price"]
)

results.head(20)

,Actual Price,Predicted Price,Error
0,270000.0,4.974250e+05,-2.274250e+05
1,525000.0,4.669639e+05,5.803609e+04
2,990000.0,6.799097e+05,3.100903e+05
3,3800000.0,1.949911e+06,1.850089e+06
4,229999.0,1.410690e+05,8.892995e+04
5,325000.0,2.855185e+05,3.948150e+04
6,210000.0,1.426002e+05,6.739981e+04
7,500000.0,7.163642e+05,-2.163642e+05
8,790000.0,9.647753e+05,-1.747753e+05
9,360000.0,2.851278e+05,7.487216e+04


In [35]:
query = """
SELECT
    id,
    car_name,
    year,
    km_driven,
    fuel,
    seller_type,
    transmission,
    owner,
    selling_price
FROM cars;
"""

df_v2 = pd.read_sql(query, connection)

print(df_v2.shape)
df_v2.head()

(3577, 9)


C:\Users\HP\AppData\Local\Temp\ipykernel_22968\1089843448.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_v2 = pd.read_sql(query, connection)


,id,car_name,year,km_driven,fuel,seller_type,transmission,owner,selling_price
0,1,Maruti 800 AC,2007,70000,Petrol,Individual,Manual,First Owner,60000.0
1,2,Maruti Wagon R LXI Minor,2007,50000,Petrol,Individual,Manual,First Owner,135000.0
2,3,Hyundai Verna 1.6 SX,2012,100000,Diesel,Individual,Manual,First Owner,600000.0
3,4,Datsun RediGO T Option,2017,46000,Petrol,Individual,Manual,First Owner,250000.0
4,5,Honda Amaze VX i-DTEC,2014,141000,Diesel,Individual,Manual,Second Owner,450000.0


In [36]:
df_v2["car_brand"] = (
    df_v2["car_name"]
    .str.strip()
    .str.split()
    .str[0]
    .str.title()
)

print(df_v2[["car_name", "car_brand"]].head(15))

                           car_name  car_brand
0                     Maruti 800 AC     Maruti
1          Maruti Wagon R LXI Minor     Maruti
2              Hyundai Verna 1.6 SX    Hyundai
3            Datsun RediGO T Option     Datsun
4             Honda Amaze VX i-DTEC      Honda
5              Maruti Alto LX BSIII     Maruti
6         Hyundai Xcent 1.2 Kappa S    Hyundai
7          Tata Indigo Grand Petrol       Tata
8          Hyundai Creta 1.6 VTVT S    Hyundai
9          Maruti Celerio Green VXI     Maruti
10          Chevrolet Sail 1.2 Base  Chevrolet
11         Tata Indigo Grand Petrol       Tata
12  Toyota Corolla Altis 1.8 VL CVT     Toyota
13             Maruti Ciaz VXi Plus     Maruti
14      Hyundai Venue SX Opt Diesel    Hyundai


In [37]:
df_v2["car_age"]=2026-df["year"]

In [38]:
df_v2["car_age"]=2026-df["year"]
df_v2.head()

,id,car_name,year,km_driven,fuel,seller_type,transmission,owner,selling_price,car_brand,car_age
0,1,Maruti 800 AC,2007,70000,Petrol,Individual,Manual,First Owner,60000.0,Maruti,19
1,2,Maruti Wagon R LXI Minor,2007,50000,Petrol,Individual,Manual,First Owner,135000.0,Maruti,19
2,3,Hyundai Verna 1.6 SX,2012,100000,Diesel,Individual,Manual,First Owner,600000.0,Hyundai,14
3,4,Datsun RediGO T Option,2017,46000,Petrol,Individual,Manual,First Owner,250000.0,Datsun,9
4,5,Honda Amaze VX i-DTEC,2014,141000,Diesel,Individual,Manual,Second Owner,450000.0,Honda,12


In [39]:
numeric_features=[ "year",
                       "km_driven" ,
                       "car_age"
]

categorical_features=["car_name",
                           "fuel",
                           "seller_type",
                           "transmission",
                           "owner",
                           "car_brand"]

X_v2 = df_v2[numeric_features + categorical_features].copy()
y_v2 = df_v2["selling_price"].copy()

X_v2=pd.get_dummies(
    X_v2,
    columns=categorical_features,
    drop_first=True,
    dtype=int
)

print( X_v2.shape[1])
X_v2.head()


1532


,year,km_driven,car_age,car_name_Ambassador Classic 2000 Dsz,car_name_Ambassador Grand 1800 ISZ MPFI PW CL,car_name_Audi A4 1.8 TFSI,car_name_Audi A4 2.0 TDI,car_name_Audi A4 2.0 TDI 177 Bhp Premium Plus,car_name_Audi A4 3.0 TDI Quattro,car_name_Audi A4 30 TFSI Technology,...,car_brand_Mg,car_brand_Mitsubishi,car_brand_Nissan,car_brand_Opelcorsa,car_brand_Renault,car_brand_Skoda,car_brand_Tata,car_brand_Toyota,car_brand_Volkswagen,car_brand_Volvo
0,2007,70000,19,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2007,50000,19,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2012,100000,14,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,2017,46000,9,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,2014,141000,12,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [40]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train_v2,X_test_v2,y_train_v2,y_test_v2=train_test_split(
    X_v2,
    y_v2,
    test_size=0.20,
    random_state=42
)

scaler_v2 = StandardScaler()
X_train_v2=scaler_v2.fit_transform(X_train_v2)
X_test_v2 = scaler_v2.transform(X_test_v2)

In [41]:
model_v2=Sequential([
    Dense(128 , activation="relu",input_shape=(X_train_v2.shape[1],)),
    Dense(64 , activation="relu"),
    Dense(32 , activation="relu"),
    Dense(1)
])

model_v2.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

history_v2 = model_v2.fit(
    X_train_v2,
    y_train_v2,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1

)

Epoch 1/100


C:\Users\HP\AppData\Roaming\Python\Python314\site-packages\keras\src\layers\core\dense.py:102: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 452640800768.0000 - mae: 471139.5938 - val_loss: 519519993856.0000 - val_mae: 473748.2188
Epoch 2/100
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 450423816192.0000 - mae: 468988.6562 - val_loss: 514733768704.0000 - val_mae: 468994.0000
Epoch 3/100
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 430574796800.0000 - mae: 449685.4688 - val_loss: 486953746432.0000 - val_mae: 440046.7188
Epoch 4/100
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 362006511616.0000 - mae: 376420.1250 - val_loss: 421854806016.0000 - val_mae: 361654.9375
Epoch 5/100
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 259580248064.0000 - mae: 266728.7812 - val_loss: 370255724544.0000 - val_mae: 320695.1562
Epoch 6/100
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 202823778304.0000 - mae: 224493.2656 - val_loss: 356426678272.0000 - val_mae: 317795.2812
Epoch 7/100
72/72 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 180421902336.0000 - mae: 206828.8125 - val_loss: 3464188067

In [42]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

y_pred_v2 = model_v2.predict(X_test_v2).flatten()

mae_v2 = mean_absolute_error(y_test_v2, y_pred_v2)
mse_v2 = mean_squared_error(y_test_v2, y_pred_v2)
rmse_v2 = np.sqrt(mse_v2)
r2_v2 = r2_score(y_test_v2, y_pred_v2)

print("ANN Version 2")
print("---------------------------")
print("MAE :", mae_v2)
print("MSE :", mse_v2)
print("RMSE:", rmse_v2)
print("R²  :", r2_v2)

23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
ANN Version 2
---------------------------
MAE : 395155.5744650217
MSE : 514880336395.8
RMSE: 717551.6262930494
R²  : -0.5983453101208338


In [43]:


print("\nNumber of unique brands:")
print(df_v2["car_brand"].nunique())




Number of unique brands:
29


In [44]:
print("\nMost common brands:")
print(df_v2["car_brand"].value_counts().head(20))


Most common brands:
car_brand
Maruti           1072
Hyundai           637
Mahindra          328
Tata              308
Ford              220
Honda             216
Toyota            170
Chevrolet         151
Renault           110
Volkswagen         93
Nissan             52
Skoda              49
Fiat               32
Audi               31
Datsun             29
Bmw                25
Mercedes-Benz      21
Jaguar              5
Mitsubishi          5
Land                5
Name: count, dtype: int64


In [45]:
print(df_v2["selling_price"].describe())

print("\nMissing values:")
print(df_v2.isnull().sum())

print("\nDuplicate rows:")
print(df_v2.duplicated().sum())

count    3.577000e+03
mean     4.739125e+05
std      5.093018e+05
min      2.000000e+04
25%      2.000000e+05
50%      3.500000e+05
75%      6.000000e+05
max      8.900000e+06
Name: selling_price, dtype: float64

Missing values:
id               0
car_name         0
year             0
km_driven        0
fuel             0
seller_type      0
transmission     0
owner            0
selling_price    0
car_brand        0
car_age          0
dtype: int64

Duplicate rows:
0


In [46]:
print("Training price statistics:")
print(y_train_v2.describe())

print("\nTesting price statistics:")
print(y_test_v2.describe())

Training price statistics:
count    2.861000e+03
mean     4.717467e+05
std      4.936159e+05
min      2.000000e+04
25%      2.000000e+05
50%      3.500000e+05
75%      5.900000e+05
max      8.900000e+06
Name: selling_price, dtype: float64

Testing price statistics:
count    7.160000e+02
mean     4.825667e+05
std      5.679647e+05
min      3.500000e+04
25%      2.000000e+05
50%      3.500000e+05
75%      6.000000e+05
max      8.150000e+06
Name: selling_price, dtype: float64


In [47]:
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Select input features
numeric_features = ["car_age", "km_driven"]
categorical_features = [
    "car_brand", "fuel", "seller_type", "transmission", "owner"
]

X_v3 = df_v2[numeric_features + categorical_features].copy()
y_v3 = df_v2["selling_price"].copy()

# Convert categorical columns into numeric columns
X_v3 = pd.get_dummies(
    X_v3,
    columns=categorical_features,
    drop_first=True,
    dtype=int
)

print("Input shape:", X_v3.shape)
print("Target shape:", y_v3.shape)
print("Missing values:", X_v3.isnull().sum().sum())

Input shape: (3577, 41)
Target shape: (3577,)
Missing values: 0


In [48]:
X_train_v3, X_test_v3, y_train_v3, y_test_v3 = train_test_split(
    X_v3, y_v3, test_size=0.20, random_state=42
)

# Scale input features
scaler_v3 = StandardScaler()

X_train_v3 = scaler_v3.fit_transform(X_train_v3)
X_test_v3 = scaler_v3.transform(X_test_v3)

# Transform target prices using logarithms
y_train_log_v3 = np.log1p(y_train_v3)
y_test_log_v3 = np.log1p(y_test_v3)

print("Training rows:", len(X_train_v3))
print("Testing rows:", len(X_test_v3))
print("Data preparation complete!")

Training rows: 2861
Testing rows: 716
Data preparation complete!


In [65]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import EarlyStopping

# Set random seed for reproducibility
tf.keras.utils.set_random_seed(42)

# Build ANN model
model_v3 = Sequential([
    Input(shape=(X_train_v3.shape[1],)),
    Dense(1000, activation="relu"),
    Dense(500, activation="relu"),
    Dense(360, activation="relu"),
    Dense(128, activation="relu"),
    Dense(64, activation="relu"),
    Dense(32, activation="relu"),
    Dense(1)
])

# Compile the model
model_v3.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

# Stop training when validation loss stops improving
early_stop_v3 = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

model_v3.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_26 (Dense)                │ (None, 1000)           │        42,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 500)            │       500,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_28 (Dense)                │ (None, 360)            │       180,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_29 (Dense)                │ (None, 128)            │        46,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_30 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_31 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_32 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 779,437 (2.97 MB)

 Trainable params: 779,437 (2.97 MB)

 Non-trainable params: 0 (0.00 B)

In [73]:
history_v3 = model_v3.fit(
    X_train_v3,
    y_train_log_v3,
    validation_split=0.2,
    epochs=1000,
    batch_size=100,
    callbacks=[early_stop_v3],
    verbose=1
)


Epoch 1/1000
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - loss: 0.5132 - mae: 0.5664 - val_loss: 0.2459 - val_mae: 0.3871
Epoch 2/1000
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.1835 - mae: 0.3236 - val_loss: 0.1788 - val_mae: 0.3233
Epoch 3/1000
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.1576 - mae: 0.3007 - val_loss: 0.1625 - val_mae: 0.3082
Epoch 4/1000
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.1443 - mae: 0.2914 - val_loss: 0.1581 - val_mae: 0.3041
Epoch 5/1000
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 0.1386 - mae: 0.2869 - val_loss: 0.1555 - val_mae: 0.3013
Epoch 6/1000
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.1355 - mae: 0.2852 - val_loss: 0.1544 - val_mae: 0.3008
Epoch 7/1000
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.1330 - mae: 0.2832 - val_loss: 0.1532 - val_mae: 0.2994
Epoch 8/1000
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 0.1315 - mae: 0.2820 - val_loss: 0.1533 - val_mae: 0.2999
Epoch 9/1000
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/

In [74]:
# Predict prices in log-transformed units
y_pred_log_v3 = model_v3.predict(X_test_v3).flatten()

# Convert predictions back to rupees
y_pred_v3 = np.expm1(y_pred_log_v3)

# Prevent negative predicted prices
y_pred_v3 = np.maximum(y_pred_v3, 0)

print("First 10 actual prices:")
print(y_test_v3.iloc[:10].values)

print("\nFirst 10 predicted prices:")
print(np.round(y_pred_v3[:10], 2))

23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
First 10 actual prices:
[ 270000.  525000.  990000. 3800000.  229999.  325000.  210000.  500000.
  790000.  360000.]

First 10 predicted prices:
[4.1846391e+05 4.0055125e+05 1.1207570e+06 2.5094385e+06 2.1909850e+05
 3.1497888e+05 1.5295380e+05 4.1339544e+05 9.2983769e+05 4.0659291e+05]


In [85]:
mae_v3 = mean_absolute_error(y_test_v3, y_pred_v3)
mse_v3 = mean_squared_error(y_test_v3, y_pred_v3)
rmse_v3 = np.sqrt(mse_v3)
r2_v3 = r2_score(y_test_v3, y_pred_v3)

print(f"MAE  : ₹{mae_v3:,.2f}")
print(f"MSE  : {mse_v3:,.2f}")
print(f"RMSE : ₹{rmse_v3:,.2f}")
print(f"R²   : {r2_v3:.4f}")

MAE  : ₹143,838.49
MSE  : 122,250,290,832.66
RMSE : ₹349,643.09
R²   : 0.6205
